[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Steve-Falkovsky/Hypencoder-Entity-Linking/blob/main/notebooks/Bi-encoder_fine-tune.ipynb)

### Load the base Bi-Encoder model (SapBERT):

In [ ]:
from sentence_transformers import SentenceTransformer, models, losses

model_name = 'cambridgeltl/SapBERT-from-PubMedBERT-fulltext'
word_embedding_model = models.Transformer(model_name)
pooling_model = models.Pooling(
    word_embedding_model.get_word_embedding_dimension(),
    pooling_mode='cls'
)
model = SentenceTransformer(modules=[word_embedding_model, pooling_model])
tokenizer = model.tokenizer

In [ ]:
import torch

# Setup the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")  # This should say 'cuda'

model.to(device)

### Load the dataset:

In [ ]:
from datasets import load_dataset

dataset = load_dataset("Stevenf232/BC5CDR_MeSH2015_complete")
train_dataset = dataset["train"]
validation_dataset = dataset["validation"]

In [ ]:
train_dataset[0]

### Pre-process the data:

In [ ]:
from datasets import Dataset

# util function to get context window around the mention in the text
!wget -q https://raw.githubusercontent.com/Steve-Falkovsky/Hypencoder-Entity-Linking/main/colab_utils/text_preprocessing.py
from text_preprocessing import get_mt_window

def prepare_sentence_pair_dataset(dataset_entry):
    """
    Prepares a single entry from the raw dataset into the format
    expected by SentenceTransformerTrainer (sentence1, sentence2, label).
    """
    m = dataset_entry["mention"]
    mt = dataset_entry["mention_text"]
    e = dataset_entry["entity"]
    d = dataset_entry["definition"]

    # Pre-process mention context and entity definition
    context_window = 128
    mt_window = get_mt_window(m, mt, context_window)
    d_processed = d[:context_window] # definition: just take first 128 chars

    dataset_entry["sentence1"] = f"{m} [SEP] {mt_window}"
    dataset_entry["sentence2"] = f"{e} [SEP] {d_processed}"
    dataset_entry["label"] = 1

    return dataset_entry

In [ ]:
# Apply the preparation function to the training and validation datasets
# `pairs` and `val_pairs` were defined in previous cells from `dataset_dict`
train_dataset = train_dataset.map(
    prepare_sentence_pair_dataset,
    remove_columns=['mention', 'mention_text', 'entity', 'definition', 'aliases', 'id']
)
eval_dataset = validation_dataset.map(
    prepare_sentence_pair_dataset,
    remove_columns=['mention', 'mention_text', 'entity', 'definition', 'aliases', 'id']
)

print(f"Prepared train_dataset sample: {train_dataset[0]}")
print(f"Prepared eval_dataset sample: {eval_dataset[0]}")

### Training (fine-tuning) using the [Sentence transformer trainer API](https://sbert.net/docs/sentence_transformer/training_overview.html#trainer):

In [ ]:
from sentence_transformers import SentenceTransformerTrainer, losses
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

# Using ContrastiveLoss since we have explicit positive and negative pairs
loss = losses.MultipleNegativesRankingLoss(model=model)

args = SentenceTransformerTrainingArguments(
    output_dir="models/sapbert-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    fp16=True,
    eval_strategy="steps",
    eval_steps=10,
    save_strategy="steps",
    save_steps=100,
    max_steps=200,
    logging_steps=10,
    logging_dir="logs",
    report_to = ["tensorboard"]
)

trainer = SentenceTransformerTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    loss=loss,
    args=args
)
trainer.train()

In [ ]:
# Load the TensorBoard notebook extension
%load_ext tensorboard

In [ ]:
# visualise training results
%tensorboard --logdir logs

## Download the data to file

In [ ]:
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
import pandas as pd

# Path to log directory
acc = EventAccumulator('logs')
acc.Reload()

data = acc.Scalars('train/loss') # tag name
df = pd.DataFrame(data)
df.to_csv('training_loss.csv')

In [ ]:
from google.colab import userdata
from huggingface_hub import login

# Login into Hugging Face Hub
hf_token = userdata.get('HF_TOKEN') # If you are running inside a Google Colab
login(hf_token)


# create a new huggingFace repo for this model
model.push_to_hub("SapBERT_MultipleNegativesRankingLoss_BC5CDR_Context")
